# blackbaud_solicit_code_webhook_events_sync
Processes Blackbaud webhook events for solicit code changes and syncs
communication preferences back to Domo.

In [ ]:
%run user_configuration.ipynb
%run renxt_core.ipynb
import pandas as pd


In [ ]:
LOOKBACK_DAYS       = 1
SOLICIT_EVENT_TYPES = [
    "com.blackbaud.constituent.solicitcode.add.v1",
    "com.blackbaud.constituent.solicitcode.change.v1",
    "com.blackbaud.constituent.solicitcode.delete.v1",
]
WEBHOOK_SOURCE_DATASET  = "raw_blackbaud_webhook_events"
COMM_PREF_URL_TPL       = API_BASE + "/constituent/v1/constituents/{cid}/communicationpreferences"
COMM_PREF_OUTPUT_DATASET = "raw_re_communicationpreferences"  # replace with UUID


In [ ]:
raw_events = domo.read_dataframe(WEBHOOK_SOURCE_DATASET, query="SELECT * FROM table")
raw_events["event_time_utc"] = pd.to_datetime(raw_events["event_time_utc"],
                                               errors="coerce", utc=True)
cutoff     = pd.Timestamp.now(tz="UTC") - pd.Timedelta(days=LOOKBACK_DAYS)
solicit_df = (
    raw_events[raw_events["event_type"].isin(SOLICIT_EVENT_TYPES)]
    .dropna(subset=["event_time_utc"])
    .loc[lambda d: d["event_time_utc"] >= cutoff]
    .reset_index(drop=True)
)
print(f"Total events loaded          : {len(raw_events):,}")
print(f"Solicit events (last {LOOKBACK_DAYS}d)   : {len(solicit_df):,}")


In [ ]:
const_ids = [cid for cid in solicit_df["entity_id"].dropna().unique() if pd.notna(cid)]
print(f"Unique constituent IDs: {len(const_ids):,}")

token_mgr = TokenManager(interactive=False)
sess      = requests.Session()
results   = []

for cid in const_ids:
    resp = api_request_with_auth(
        "GET", COMM_PREF_URL_TPL.format(cid=cid),
        token_mgr=token_mgr, session=sess,
    )
    results.append({"constituent_id": cid, "_status": resp.status_code,
                    "data": resp.json() if resp.ok else None})

print(f"Fetched {sum(r['_status']==200 for r in results):,}/{len(results):,} successfully")


In [ ]:
rows = []
for r in results:
    if r["_status"] != 200 or not r["data"]:
        continue
    data  = r["data"]
    prefs = data if isinstance(data, list) else data.get("value", [])
    for p in prefs:
        if isinstance(p, dict):
            p["constituent_id"] = r["constituent_id"]
            rows.append(p)

comm_pref_df = pd.DataFrame(rows) if rows else pd.DataFrame(
    columns=["constituent_id", "id", "solicit_code", "start", "end"])
print(f"Parsed {len(comm_pref_df):,} comm preference records")


In [ ]:
if not comm_pref_df.empty:
    try:
        existing_df = domo.read_dataframe(COMM_PREF_OUTPUT_DATASET,
                                          query="SELECT * FROM table")
        combined_df = (
            pd.concat([existing_df, comm_pref_df], ignore_index=True)
            .drop_duplicates(subset=["constituent_id", "id"], keep="last")
            .reset_index(drop=True)
        )
    except Exception:
        combined_df = comm_pref_df.copy()

    df_out = domo_safe_cast(combined_df)
    domo.write_dataframe(df_out, dataset=COMM_PREF_OUTPUT_DATASET, update_method="REPLACE")
    print(f"✅ Wrote {len(df_out):,} rows → {COMM_PREF_OUTPUT_DATASET}")
else:
    print("⚠️  No comm preference records to write")
